# 02 — Clean injury events and build episodes

This notebook turns raw transactions, schedules, appearances, and biographies into the core injury-episode table. Read [`context/CLEANING_CONTEXT.md`](../context/CLEANING_CONTEXT.md) for the complete rules and audit decisions.

In [ ]:
from pathlib import Path
import subprocess
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
AUDIT_DIR = PROJECT_ROOT / 'data' / 'audits'
SRC_DIR = PROJECT_ROOT / 'src'
for directory in [RAW_DIR, PROCESSED_DIR, AUDIT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

## Build the episode table

The tested builder handles name/team normalization, injury filtering, body-part extraction, episode consolidation, schedule-based missed-game counts, demographics, and future-major-injury flags. Set `RUN_BUILD` to `True` after the raw-file readiness table is complete.

In [ ]:
required_raw = [
    'kaggle_nba_injury_stats_1951_2023.csv',
    'prosportstransactions_scrape_missedgames_2010_2019.csv',
    'prosportstransactions_scrape_IRL_2010_2019.csv',
    'all_teams_schedule_2010_2020.csv',
]
readiness = pd.DataFrame({
    'Raw file': required_raw,
    'Present': [(RAW_DIR / name).exists() for name in required_raw],
})
readiness

In [ ]:
RUN_BUILD = False

if RUN_BUILD:
    if not readiness['Present'].all():
        missing = readiness.loc[~readiness['Present'], 'Raw file'].tolist()
        raise FileNotFoundError(f'Missing required raw files: {missing}')
    subprocess.run([sys.executable, str(SRC_DIR / 'build_dataset.py')], cwd=PROJECT_ROOT, check=True)

## Load and inspect the result

In [ ]:
output_path = PROCESSED_DIR / 'nba_injuries.csv'
injuries = pd.read_csv(output_path, parse_dates=['Date', 'Next Major Injury Date']) if output_path.exists() else None
if injuries is None:
    print('No processed injury table yet. Add the raw files and run the build cell.')
else:
    display(injuries.head())
    print(f'{len(injuries):,} injury episodes from {injuries.Date.min().date()} to {injuries.Date.max().date()}')

## Cleaning invariants

Unknown durations remain unknown, MCL maps to knee, and only a value of at least 30 games or `season ending` qualifies as major.

In [ ]:
REQUIRED_COLUMNS = [
    'Player', 'Team', 'Date', 'Body Part', 'Injury Description (Raw)',
    'number of missed games', 'Major Injury in Next 3 Years',
    'Major Injury in Next 3 Years - Same Body Part', 'Next Major Injury Date',
    'Height (inches)', 'Weight (lb)', 'Age at Injury',
]

def is_major(value):
    if isinstance(value, str) and value.strip().lower() == 'season ending':
        return True
    numeric = pd.to_numeric(pd.Series([value]), errors='coerce').iloc[0]
    return pd.notna(numeric) and numeric >= 30

assert is_major(30)
assert not is_major(29)
assert is_major('season ending')
assert not is_major('unknown')

if injuries is not None:
    assert set(REQUIRED_COLUMNS).issubset(injuries.columns)
    assert not injuries.duplicated(['Player', 'Team', 'Date']).any()
    mcl = injuries['Injury Description (Raw)'].str.contains(r'\bMCL\b', case=False, na=False)
    assert injuries.loc[mcl, 'Body Part'].str.lower().eq('knee').all()
    numeric = pd.to_numeric(injuries['number of missed games'], errors='coerce')
    assert numeric.dropna().between(0, 82).all()
    print('Core cleaning invariants: PASS')

## Regression audit

These cases caught the original transaction-row-count error and anchor the expected schedule-derived values.

In [ ]:
regressions = pd.read_csv(AUDIT_DIR / 'regression_cases.csv', parse_dates=['Date'])
if injuries is None:
    display(regressions)
else:
    observed = injuries.merge(regressions, on=['Player', 'Team', 'Date'], how='right')
    observed['Observed missed games'] = pd.to_numeric(observed['number of missed games'], errors='coerce')
    observed['PASS'] = observed['Observed missed games'].eq(observed['Expected missed games'])
    display(observed[['Player', 'Date', 'Expected missed games', 'Observed missed games', 'PASS', 'Audit note']])
    assert observed['PASS'].all()

## Optional independent audits

After a successful build, run `src/audit_kaggle.py`, `src/audit_cases.py`, and `src/audit_future_metrics.py`. Audit failures should block downstream analysis; unresolved endpoints should remain `unknown` rather than being imputed.